# Gestión de Clientes y Proyectos – SEO Pipeline 2025

Este notebook permite:
- Crear nuevos clientes con todas sus credenciales
- Crear proyectos asociados a un cliente
- Listar y seleccionar cliente/proyecto activo
- Todo se guarda automáticamente en `data/clients.json` y `data/projects.json`

Una vez configurado, nunca más tendrás que volver a introducir las API keys.

In [ ]:
# Celda 1 — Setup inicial
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive

import sys
sys.path.append('/content/drive/MyDrive/seo_pipeline')

from seo_pipeline.config import get_config
from seo_pipeline.utils.logging import setup_logging
setup_logging(level="INFO")

cfg = get_config()
print("Configuración cargada correctamente")

In [ ]:
# Celda 2 — Crear NUEVO CLIENTE (ejecutar y rellenar)
from pydantic import SecretStr
import getpass

print("=== ALTA DE NUEVO CLIENTE ===\n")

client_id = input("ID único del cliente (ej: acme_corp): ").strip()
name = input("Nombre del cliente: ").strip()

print("\nCredenciales (puedes dejar en blanco si no usas esa herramienta):")
semrush_token = getpass.getpass("SEMrush API token: ").strip() or None
serpapi_key = getpass.getpass("SerpAPI key: ").strip() or None
openai_key = getpass.getpass("OpenAI API key: ").strip() or None
gsc_sa = input("Ruta al JSON Service Account GSC (ej: credentials/gsc_sa.json): ").strip() or None
sheets_sa = input("Ruta al JSON Service Account Sheets (ej: credentials/sheets_sa.json): ").strip() or None

# Crear y guardar
from seo_pipeline.config import ClientConfig
new_client = ClientConfig(
    client_id=client_id,
    name=name,
    semrush_token=semrush_token,
    serpapi_key=serpapi_key,
    openai_key=openai_key,
    gsc_sa_path=gsc_sa,
    sheets_sa_path=sheets_sa,
    default_database="es",
    default_gl="es",
    default_hl="es-es"
)

cfg.clients[client_id] = new_client
cfg.save_clients()
print(f"\n✓ Cliente '{name}' creado y guardado correctamente.")

In [ ]:
# Celda 3 — Crear NUEVO PROYECTO (vinculado a un cliente)
print("Clientes disponibles:")
for cid, c in cfg.clients.items():
    print(f"  • {cid} → {c.name}")

client_id = input("\nID del cliente al que pertenece este proyecto: ").strip()
if client_id not in cfg.clients:
    raise ValueError("Cliente no encontrado")

project_id = input("ID único del proyecto (ej: blog_es): ").strip()
name = input("Nombre del proyecto: ").strip()
base_domain = input("Dominio principal (ej: ejemplo.com): ").strip()
gsc_property = input("URL completa de la propiedad GSC (ej: https://ejemplo.com/): ").strip()
sheets_id = input("ID o URL completa de la hoja Google Sheets principal: ").strip()

from seo_pipeline.config import ProjectConfig
new_project = ProjectConfig(
    project_id=project_id,
    client_id=client_id,
    name=name,
    base_domain=base_domain,
    gsc_property=gsc_property,
    sheets_id=sheets_id,
    output_dir="outputs"
)

cfg.projects[project_id] = new_project
cfg.save_projects()
print(f"\n✓ Proyecto '{name}' creado y vinculado al cliente '{cfg.clients[client_id].name}'")

In [ ]:
# Celda 4 — Seleccionar cliente y proyecto activo (para ejecuciones futuras)
print("=== SELECCIÓN DE CLIENTE/PROYECTO ACTIVO ===\n")
print("Clientes disponibles:")
for cid, c in cfg.clients.items():
    active = "(ACTIVO)" if cfg.active_client and cfg.active_client.client_id == cid else ""
    print(f"  • {cid} → {c.name} {active}")

client_id = input("\nEscribe el ID del cliente que quieres activar: ").strip()
if cfg.set_active_client(client_id):
    print(f"\n✓ Cliente activo: {cfg.active_client.name}")
else:
    print("Cliente no encontrado")

print("\nProyectos disponibles para este cliente:")
for pid, p in cfg.projects.items():
    if p.client_id == client_id:
        active = "(ACTIVO)" if cfg.active_project and cfg.active_project.project_id == pid else ""
        print(f"  • {pid} → {p.name} {active}")

project_id = input("\nEscribe el ID del proyecto que quieres activar: ").strip()
if cfg.set_active_project(project_id):
    print(f"\n✓ Proyecto activo: {cfg.active_project.name}")
    print(f"   Dominio: {cfg.active_project.base_domain}")
    print(f"   GSC: {cfg.active_project.gsc_property}")
    print(f"   Sheets: {cfg.active_project.sheets_id}")
else:
    print("Proyecto no encontrado o no pertenece al cliente activo")

In [ ]:
# Celda 5 — Ver estado actual (ejecutar en cualquier momento)
print("=== ESTADO ACTUAL DEL PIPELINE ===\n")

if cfg.active_client:
    print(f"Cliente activo : {cfg.active_client.name} ({cfg.active_client.client_id})")
    print(f"   SEMrush : {'✓' if cfg.active_client.semrush_token else '✗'} configurado")
    print(f"   SerpAPI : {'✓' if cfg.active_client.serpapi_key else '✗'} configurado")
    print(f"   OpenAI  : {'✓' if cfg.active_client.openai_key else '✗'} configurado")
else:
    print("Ningún cliente activo")

if cfg.active_project:
    print(f"\nProyecto activo: {cfg.active_project.name} ({cfg.active_project.project_id})")
    print(f"   Dominio      : {cfg.active_project.base_domain}")
    print(f"   GSC property : {cfg.active_project.gsc_property}")
    print(f"   Google Sheets: {cfg.active_project.sheets_id}")
else:
    print("\nNingún proyecto activo")

print(f"\nDirectorio de salidas: {cfg.get_output_dir() if cfg.active_project else 'N/A'}")